In [ ]:
"""Colab-ready raw-audio regression notebook for the Drive `dataset` folder.

Architecture:
- Raw waveform input from `dataset/sample_*.wav`
- Deep residual 1D CNN with small kernels, downsampling stages, squeeze-excite, and dense regression head
- Mixed precision and tf.data pipeline tuned for Google Colab GPU memory

Data flow:
- Input: `dataset/parameters.csv` and matching WAV files generated in Colab
- Output: trained `.keras` model, scaler, history CSV/plots, predictions CSV, and `results.json`
"""

import json
import os
from pathlib import Path

try:
    from google.colab import drive
except Exception:
    drive = None

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    Input,
    LayerNormalization,
    Multiply,
    Reshape,
    SpatialDropout1D,
)
from keras.models import Model
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import mixed_precision

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/fmsynth")
if drive is not None:
    drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

BASE_PATH = DRIVE_ROOT / "dataset"
OUTPUT_DIR = DRIVE_ROOT / "model_training_big4_fmsynth3_0_2_colab"
MODEL_NAME = "model_training_big4_fmsynth3_0_2_colab"
SEED = 42
TRAIN_FRAC = 0.75
VAL_FRAC_OF_TRAIN = 0.2
BATCH_SIZE = int(os.getenv("TRAIN_BATCH_SIZE", "16"))
EPOCHS = int(os.getenv("EPOCHS", "200"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "3e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-5"))
AUTOTUNE = tf.data.AUTOTUNE

# If you hit OOM on Colab, lower this to 8.
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.keras.utils.set_random_seed(SEED)
except Exception:
    pass

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(exc)
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision policy:", mixed_precision.global_policy())
else:
    print("GPU not found; running in float32.")


def to_json_scalar(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


# -----------------------------------------------------------------------------
# Load dataset metadata and targets
# -----------------------------------------------------------------------------
meta = {}
meta_path = Path(BASE_PATH) / "meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))

params_path = Path(BASE_PATH) / "parameters.csv"
if not params_path.exists():
    raise FileNotFoundError(
        f"{params_path} not found. Run goolge_colab/gcp_generate_dataset_big4_fmsynth3_0_1.ipynb first and make sure it wrote to /content/drive/MyDrive/fmsynth/dataset."
    )

target_raw = pd.read_csv(params_path)
tamanho_dataset = int(meta.get("tamanho_dataset", len(target_raw)))
if tamanho_dataset < len(target_raw):
    target_raw = target_raw.iloc[:tamanho_dataset].copy()
    tamanho_dataset = len(target_raw)

sample_rate_out = int(meta.get("sample_rate_out", 16000))
duration_seconds = float(meta.get("duracao_amostras", 4.0))
target_samples = int(round(sample_rate_out * duration_seconds))
print(
    f"Dataset size={tamanho_dataset}, sample_rate={sample_rate_out}, "
    f"duration={duration_seconds}, samples_per_file={target_samples}"
)

if "id" in target_raw.columns:
    sample_ids = target_raw["id"].astype(int).tolist()
else:
    sample_ids = list(range(tamanho_dataset))

audio_paths = np.array([str(Path(BASE_PATH) / f"sample_{sid}.wav") for sid in sample_ids])
missing_paths = [path for path in audio_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing audio file: {missing_paths[0]}")

# Encode categorical and boolean columns in a reproducible way.
categorical_maps = {}
target_encoded = target_raw.copy()
for col in target_encoded.columns:
    if target_encoded[col].dtype == object:
        cat = pd.Categorical(target_encoded[col])
        categorical_maps[col] = [str(x) for x in cat.categories]
        target_encoded[col] = cat.codes.astype(np.int32)
    elif target_encoded[col].dtype == bool:
        target_encoded[col] = target_encoded[col].astype(np.int32)

if "id" in target_encoded.columns:
    target_encoded = target_encoded.drop(columns=["id"])

constant_targets = {}
for col in list(target_encoded.columns):
    if target_encoded[col].nunique(dropna=False) <= 1:
        constant_targets[col] = to_json_scalar(target_encoded[col].iloc[0])

target_model = target_encoded.drop(columns=list(constant_targets.keys())).copy()
if target_model.empty:
    raise ValueError("All targets were removed as constants.")

target_columns = list(target_model.columns)
categorical_cols = [col for col in target_columns if col in categorical_maps]
numeric_cols = [col for col in target_columns if col not in categorical_cols]

# log2 helps the model learn multiplicative parameters with smaller dynamic range.
log2_cols = [
    col
    for col in [
        "frequencia_base",
        "ratio1",
        "ratio2",
        "ratio3",
        "ratio4",
        "ratio5",
        "ratio_carrier",
    ]
    if col in target_columns
]

target_transformed = target_model.copy().astype(np.float32)
for col in log2_cols:
    values = target_transformed[col].to_numpy(dtype=np.float32)
    if np.any(values <= 0):
        raise ValueError(f"Column {col} contains non-positive values, cannot apply log2.")
    target_transformed[col] = np.log2(values)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(target_transformed.to_numpy(dtype=np.float32)).astype(np.float32)

print("Target columns:", len(target_columns))
print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))
print("Removed constant targets:", constant_targets)
print("Log2 columns:", log2_cols)


# -----------------------------------------------------------------------------
# Train/validation/test split
# -----------------------------------------------------------------------------
all_indices = np.arange(len(target_model))
train_idx = target_model.sample(frac=TRAIN_FRAC, random_state=SEED).index.to_numpy()
test_idx = np.setdiff1d(all_indices, train_idx, assume_unique=False)

train_paths = audio_paths[train_idx]
test_paths = audio_paths[test_idx]
train_scaled = y_scaled[train_idx]
test_scaled = y_scaled[test_idx]
train_raw = target_model.iloc[train_idx].reset_index(drop=True)
test_raw = target_model.iloc[test_idx].reset_index(drop=True)

train_local_indices = np.arange(len(train_paths))
val_local_indices = train_raw.sample(frac=VAL_FRAC_OF_TRAIN, random_state=SEED).index.to_numpy()
fit_local_indices = np.setdiff1d(train_local_indices, val_local_indices, assume_unique=False)

fit_paths = train_paths[fit_local_indices]
val_paths = train_paths[val_local_indices]
fit_scaled = train_scaled[fit_local_indices]
val_scaled = train_scaled[val_local_indices]

TARGET_DIM = len(target_columns)

print(
    f"Split sizes -> fit={len(fit_paths)}, val={len(val_paths)}, test={len(test_paths)}"
)


# -----------------------------------------------------------------------------
# tf.data input pipeline
# -----------------------------------------------------------------------------

def load_example(path, label):
    audio_bytes = tf.io.read_file(path)
    waveform, _sample_rate = tf.audio.decode_wav(
        audio_bytes,
        desired_channels=1,
        desired_samples=target_samples,
    )
    waveform = tf.ensure_shape(waveform, [target_samples, 1])
    waveform = tf.cast(waveform, tf.float32)
    label = tf.ensure_shape(label, [TARGET_DIM])
    label = tf.cast(label, tf.float32)
    return waveform, label


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_example, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(fit_paths, fit_scaled, training=True)
val_ds = make_dataset(val_paths, val_scaled, training=False)
test_ds = make_dataset(test_paths, test_scaled, training=False)

for xb, yb in train_ds.take(1):
    print("Batch shapes:", xb.shape, yb.shape)


# -----------------------------------------------------------------------------
# Model definition
# -----------------------------------------------------------------------------

def residual_block(x, filters, kernel_size, stride=1, dropout=0.0, name="res"):
    shortcut = x

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv1",
    )(x)
    x = BatchNormalization(name=f"{name}_bn1")(x)
    x = Activation("gelu", name=f"{name}_act1")(x)
    if dropout > 0:
        x = SpatialDropout1D(dropout, name=f"{name}_drop1")(x)

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_conv2",
    )(x)
    x = BatchNormalization(name=f"{name}_bn2")(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv1D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{name}_skip",
        )(shortcut)
        shortcut = BatchNormalization(name=f"{name}_skip_bn")(shortcut)

    x = Add(name=f"{name}_add")([x, shortcut])
    x = Activation("gelu", name=f"{name}_out")(x)
    return x


def squeeze_excite(x, reduction=16, name="se"):
    channels = int(x.shape[-1])
    hidden_units = max(channels // reduction, 16)
    se = GlobalAveragePooling1D(name=f"{name}_gap")(x)
    se = Dense(hidden_units, activation="gelu", name=f"{name}_dense1")(se)
    se = Dense(channels, activation="sigmoid", name=f"{name}_dense2")(se)
    se = Reshape((1, channels), name=f"{name}_reshape")(se)
    return Multiply(name=f"{name}_scale")([x, se])


def build_model(input_len, output_dims):
    inputs = Input(shape=(input_len, 1), name="audio_input")
    x = BatchNormalization(name="input_bn")(inputs)

    # Stem
    x = Conv1D(
        filters=32,
        kernel_size=7,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv",
    )(x)
    x = BatchNormalization(name="stem_bn")(x)
    x = Activation("gelu", name="stem_act")(x)

    # Stage 1
    x = residual_block(x, 32, 5, stride=1, dropout=0.02, name="stage1_block1")
    x = residual_block(x, 32, 5, stride=1, dropout=0.02, name="stage1_block2")

    # Stage 2
    x = residual_block(x, 48, 5, stride=2, dropout=0.03, name="stage2_down")
    x = residual_block(x, 48, 5, stride=1, dropout=0.03, name="stage2_block1")
    x = residual_block(x, 48, 5, stride=1, dropout=0.03, name="stage2_block2")

    # Stage 3
    x = residual_block(x, 64, 3, stride=2, dropout=0.04, name="stage3_down")
    x = residual_block(x, 64, 3, stride=1, dropout=0.04, name="stage3_block1")
    x = residual_block(x, 64, 3, stride=1, dropout=0.04, name="stage3_block2")

    # Stage 4
    x = residual_block(x, 96, 3, stride=2, dropout=0.05, name="stage4_down")
    x = residual_block(x, 96, 3, stride=1, dropout=0.05, name="stage4_block1")
    x = residual_block(x, 96, 3, stride=1, dropout=0.05, name="stage4_block2")

    # Stage 5
    x = residual_block(x, 128, 3, stride=2, dropout=0.08, name="stage5_down")
    x = residual_block(x, 128, 3, stride=1, dropout=0.08, name="stage5_block1")
    x = residual_block(x, 128, 3, stride=1, dropout=0.08, name="stage5_block2")

    # Stage 6
    x = residual_block(x, 192, 3, stride=2, dropout=0.10, name="stage6_down")
    x = residual_block(x, 192, 3, stride=1, dropout=0.10, name="stage6_block1")
    x = residual_block(x, 192, 3, stride=1, dropout=0.10, name="stage6_block2")

    x = squeeze_excite(x, reduction=16, name="tail_se")
    gap = GlobalAveragePooling1D(name="gap")(x)
    gmp = GlobalMaxPooling1D(name="gmp")(x)
    x = Concatenate(name="pool_concat")([gap, gmp])
    x = LayerNormalization(name="head_ln")(x)

    x = Dense(1024, activation="gelu", name="head_dense_1")(x)
    x = Dropout(0.40, name="head_drop_1")(x)
    x = Dense(512, activation="gelu", name="head_dense_2")(x)
    x = Dropout(0.25, name="head_drop_2")(x)
    x = Dense(256, activation="gelu", name="head_dense_3")(x)
    x = Dropout(0.15, name="head_drop_3")(x)

    outputs = Dense(
        output_dims,
        activation="linear",
        dtype="float32",
        name="targets",
    )(x)

    return Model(inputs, outputs, name="deep_residual_cnn_regressor")


model = build_model(target_samples, TARGET_DIM)
model.summary()

try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0,
    )
except AttributeError:
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0,
    )

model.compile(optimizer=optimizer, loss="mse", metrics=["mae", "mse"])


# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------
best_model_path = Path(OUTPUT_DIR) / f"{MODEL_NAME}.keras"
callbacks = [
    ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-6,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

hist = pd.DataFrame(history.history)
hist["epoch"] = np.arange(1, len(hist) + 1)
hist.to_csv(Path(OUTPUT_DIR) / "train_history.csv", index=False)

plt.figure(dpi=400)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.plot(hist["epoch"], hist["loss"], label="Training loss")
plt.plot(hist["epoch"], hist["val_loss"], label="Validation loss")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "train_history_loss.png", dpi=400)
plt.close()

plt.figure(dpi=400)
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.plot(hist["epoch"], hist["mae"], label="Training MAE")
plt.plot(hist["epoch"], hist["val_mae"], label="Validation MAE")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "train_history_mae.png", dpi=400)
plt.close()

plt.figure(dpi=400)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.plot(hist["epoch"], hist["mse"], label="Training MSE")
plt.plot(hist["epoch"], hist["val_mse"], label="Validation MSE")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "train_history_mse.png", dpi=400)
plt.close()


# -----------------------------------------------------------------------------
# Evaluation helpers
# -----------------------------------------------------------------------------
def inverse_predictions(pred_scaled):
    pred_transformed = scaler_y.inverse_transform(pred_scaled.astype(np.float32))
    pred_raw = pred_transformed.copy()
    for col in log2_cols:
        idx = target_columns.index(col)
        pred_raw[:, idx] = np.power(2.0, pred_raw[:, idx])
    return pred_raw


# -----------------------------------------------------------------------------
# Test set evaluation
# -----------------------------------------------------------------------------
y_pred_scaled = model.predict(test_ds, verbose=1)
y_pred_raw = inverse_predictions(y_pred_scaled)
y_true_raw = test_raw.to_numpy(dtype=np.float32)

mse = float(np.mean(np.square(y_true_raw - y_pred_raw)))
mae = float(np.mean(np.abs(y_true_raw - y_pred_raw)))
rmse = float(np.sqrt(mse))

mae_by_col = np.mean(np.abs(y_true_raw - y_pred_raw), axis=0)

categorical_accuracy = {}
for col in categorical_cols:
    idx = target_columns.index(col)
    true_codes = np.rint(y_true_raw[:, idx]).astype(int)
    pred_codes = np.rint(y_pred_raw[:, idx]).astype(int)
    pred_codes = np.clip(pred_codes, 0, len(categorical_maps[col]) - 1)
    categorical_accuracy[col] = float(np.mean(true_codes == pred_codes))

freq_mae_hz = None
freq_rmse_hz = None
if "frequencia_base" in target_columns:
    freq_idx = target_columns.index("frequencia_base")
    freq_mae_hz = float(mae_by_col[freq_idx])
    freq_rmse_hz = float(np.sqrt(np.mean(np.square(y_true_raw[:, freq_idx] - y_pred_raw[:, freq_idx]))))

ratio_cols_eval = [col for col in ["ratio1", "ratio2", "ratio3", "ratio4", "ratio5", "ratio_carrier"] if col in target_columns]
ratio_mae_mean = None
if ratio_cols_eval:
    ratio_mae_mean = float(np.mean([mae_by_col[target_columns.index(col)] for col in ratio_cols_eval]))

categorical_accuracy_mean = None
if categorical_accuracy:
    categorical_accuracy_mean = float(np.mean(list(categorical_accuracy.values())))

print(f"Test RMSE: {rmse}")
print(f"Test MSE: {mse}")
print(f"Test MAE: {mae}")
if freq_mae_hz is not None:
    print(f"Frequency MAE (Hz): {freq_mae_hz}")
if categorical_accuracy_mean is not None:
    print(f"Categorical accuracy mean: {categorical_accuracy_mean}")

pred_export = {}
for i, col in enumerate(target_columns):
    pred_export[f"true_{col}"] = y_true_raw[:, i]
    pred_export[f"pred_{col}"] = y_pred_raw[:, i]
pd.DataFrame(pred_export).to_csv(Path(OUTPUT_DIR) / "predictions_test.csv", index=False)

# Save the best model and scaler bundle.
model.save(best_model_path)
joblib.dump(scaler_y, Path(OUTPUT_DIR) / f"scaler_y_{MODEL_NAME}.save")

# -----------------------------------------------------------------------------
# Results summary
# -----------------------------------------------------------------------------
results = {
    "model_name": MODEL_NAME,
    "dataset": BASE_PATH,
    "output_dir": OUTPUT_DIR,
    "tamanho_dataset": int(tamanho_dataset),
    "split": {
        "train_frac": float(TRAIN_FRAC),
        "val_frac_of_train": float(VAL_FRAC_OF_TRAIN),
        "fit_size": int(len(fit_paths)),
        "val_size": int(len(val_paths)),
        "test_size": int(len(test_paths)),
    },
    "runtime_config": {
        "batch_size": int(BATCH_SIZE),
        "epochs": int(EPOCHS),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "mixed_precision": bool(gpus),
        "policy": mixed_precision.global_policy().name,
    },
    "model": {
        "num_params": int(model.count_params()),
    },
    "target_preprocessing": {
        "target_columns": target_columns,
        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "log2_cols": log2_cols,
        "constant_targets": constant_targets,
        "categorical_maps": categorical_maps,
    },
    "metrics": {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
    },
    "metrics_extra": {
        "freq_mae_hz": freq_mae_hz,
        "freq_rmse_hz": freq_rmse_hz,
        "ratio_mae_mean": ratio_mae_mean,
        "categorical_accuracy_mean": categorical_accuracy_mean,
    },
    "categorical_accuracy": categorical_accuracy,
    "history_last": {
        key: float(hist[key].iloc[-1]) for key in hist.columns if key != "epoch"
    },
    "best_val_loss": {
        "epoch": int(hist["val_loss"].idxmin() + 1),
        "value": float(hist["val_loss"].min()),
    },
}

with open(Path(OUTPUT_DIR) / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved results to {Path(OUTPUT_DIR) / 'results.json'}")
